# Week 5-C: Semantic Encoder Adaptation and Capacity Study

**Project:** Query-Conditioned Context Allocator (QCCA) for Efficient Retrieval-Augmented Generation  
**Parent Architecture:** SARA — Selective and Adaptive Retrieval-augmented Generation with Context Compression  
**Experiment:** Week 5-C Semantic Capacity and Encoder Adaptation Ladder (C0 Frozen, C1 Last-1 Layer, C2 Last-2 Layers)  
**Population:** QASPER TRAIN (2,088 queries) & DEV (231 queries)  
**Governance:** Strict Split Isolation. Held-out TEST set (1,309 queries) remains 100% frozen & untouched.  

---

### Research Context & Transition from Week 5-B:
Week 5 established that direct retrieved passage content contains allocation signal, but under a frozen sentence encoder (`all-MiniLM-L6-v2`) with a shallow transformer aggregator, the semantic allocator (`I3-T3`: DEV F1 ≈ 0.3810, tokens ≈ 1258.2) did not outperform the canonical handcrafted Week 4 allocator (Balanced Logistic Regression: DEV F1 ≈ 0.3842, tokens ≈ 1115.9).

### Core Research Question:
> **"Does task-specific adaptation of the semantic encoder allow the content-aware allocator to exploit retrieved-passage information more effectively than the frozen encoder tested in Week 5-B?"**

### Frozen Protocol Elements from Week 5-B:
1. **Target Formulation:** Multi-Label Epsilon-Admissibility ($T_3$): $z_k(q) = 1[\text{regret}(q, k) \le 0.01]$ over $k \in \{2, 4, 5, 6, 8\}$ via `BCEWithLogitsLoss`.
2. **Retrieval Pool:** BM25 top-10 retrieved passages.
3. **Budget Set:** $k \in \{2, 4, 5, 6, 8\}$.
4. **Information Condition:** Fixed to $I_3$ (Query + Top-10 Passages).
5. **Downstream Policy Rule:** Select smallest $k$ such that $p_k \ge \tau$, fallback to $k=8$.
6. **Aggregator Architecture:** Corrected rank embeddings initialized $\mathcal{N}(0, 0.02^2)$ with `LayerNorm(h_i + r_i)`.
7. **Threshold Grid:** $\tau \in \{0.30, 0.40, 0.50, 0.60, 0.70\}$ calibrated on INTERNAL-VAL using frozen 0.005 tolerance rule.

## Section 1: Frozen Baselines and Historical References

In [1]:
import os
import sys
import json
import logging
from pathlib import Path
import yaml

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

from src.week5.data import (
    K_CLASSES,
    NUM_CLASSES,
    ContentAllocatorDataset,
    build_w5b_splits_and_bundles,
)
from src.week5.training import set_seed
from src.week5.w5c.adaptation import (
    AdaptedContentAwareAllocator,
    audit_minilm_architecture,
    build_optimizer_param_groups,
)
from src.week5.w5c.evaluation import (
    load_c0_w5b_reference,
    load_w4_baseline_dev,
    select_tau_w5c,
    evaluate_w5c_model_on_dev,
    evaluate_allocation_policy,
    run_paper_clustered_bootstrap_comparisons,
    plot_w5c_pareto_curve,
    W4_REF_F1,
    W4_REF_TOKENS,
    C0_REF_F1,
    C0_REF_TOKENS,
)
from src.week5.w5c.training import train_adapted_allocator
from src.week5.w5c.representation_analysis import compute_representation_shift

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Load canonical C0 reference from Week 5-B
c0_ref = load_c0_w5b_reference("results/week5/content_allocator/w5b")
print("=== CANONICAL REFERENCES ===")
print(f"C0 Frozen MiniLM (I3-T3): DEV F1 = {c0_ref['mean_answer_f1']:.4f}, Tokens = {c0_ref['mean_context_tokens']:.1f}, Mean k = {c0_ref['mean_selected_k']:.2f}")
print(f"W4 Balanced LogReg Baseline: DEV F1 = {W4_REF_F1:.4f}, Tokens = {W4_REF_TOKENS:.1f}, Mean k = 4.82")
print(f"Best Static SARA (k=8): DEV F1 = 0.3950, Tokens = 1750.5, Mean k = 8.00")
print(f"Quality Oracle: DEV F1 = 0.4895, Tokens = 1021.5, Mean k = 3.53")
print(f"Cost-Aware Epsilon Oracle: DEV F1 = 0.4894, Tokens = 839.4, Mean k = 3.49")

/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== CANONICAL REFERENCES ===
C0 Frozen MiniLM (I3-T3): DEV F1 = 0.3810, Tokens = 1258.2, Mean k = 5.57
W4 Balanced LogReg Baseline: DEV F1 = 0.3842, Tokens = 1115.9, Mean k = 4.82
Best Static SARA (k=8): DEV F1 = 0.3950, Tokens = 1750.5, Mean k = 8.00
Quality Oracle: DEV F1 = 0.4895, Tokens = 1021.5, Mean k = 3.53
Cost-Aware Epsilon Oracle: DEV F1 = 0.4894, Tokens = 839.4, Mean k = 3.49


## Section 2: Experiment Configuration

In [2]:
with open("configs/week5/w5c_encoder_adaptation.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

SEEDS = config["experiment"]["seeds"]
PRIMARY_SEED = config["experiment"]["primary_seed"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPSILON = config["experiment"]["epsilon"]
TAU_GRID = config["experiment"]["tau_grid"]
TAU_TOLERANCE = config["experiment"]["tau_selection_tolerance"]

ENCODER_LR = float(config["training"]["encoder_lr"])
HEAD_LR = float(config["training"]["head_lr"])
BATCH_SIZE = int(config["training"]["batch_size"])
MAX_EPOCHS = int(config["training"]["max_epochs"])
PATIENCE = int(config["training"]["patience"])
GRAD_CLIP = float(config["training"]["grad_clip"])

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}")
print(f"Differential LRs: Encoder={ENCODER_LR}, Head/Aggregator={HEAD_LR}")
print(f"Epochs: {MAX_EPOCHS}, Patience: {PATIENCE}, Batch Size: {BATCH_SIZE}")
print(f"Epsilon: {EPSILON}, Tau Grid: {TAU_GRID}")

Device: cuda
Seeds: [42, 123, 2026]
Differential LRs: Encoder=2e-05, Head/Aggregator=0.0002
Epochs: 15, Patience: 3, Batch Size: 16
Epsilon: 0.01, Tau Grid: [0.3, 0.4, 0.5, 0.6, 0.7]


## Section 3: Split Integrity and Manifest Audit

In [3]:
# Load manifests
manifest_fit_path = Path("results/week5/content_allocator/w5b/splits/train_fit_manifest.csv")
manifest_val_path = Path("results/week5/content_allocator/w5b/splits/internal_val_manifest.csv")
assert manifest_fit_path.exists() and manifest_val_path.exists(), "Missing split manifests!"

df_fit_man = pd.read_csv(manifest_fit_path)
df_val_man = pd.read_csv(manifest_val_path)

print(f"TRAIN-FIT Manifest: {len(df_fit_man)} questions across {df_fit_man['paper_id'].nunique()} papers")
print(f"INTERNAL-VAL Manifest: {len(df_val_man)} questions across {df_val_man['paper_id'].nunique()} papers")
assert len(df_fit_man) == 1746, f"Expected 1746 FIT queries, found {len(df_fit_man)}"
assert len(df_val_man) == 342, f"Expected 342 VAL queries, found {len(df_val_man)}"

# Verify paper and question disjointness
papers_fit = set(df_fit_man["paper_id"])
papers_val = set(df_val_man["paper_id"])
qids_fit = set(df_fit_man["question_id"])
qids_val = set(df_val_man["question_id"])

assert papers_fit.isdisjoint(papers_val), "DATA LEAKAGE: Overlapping papers between FIT and VAL!"
assert qids_fit.isdisjoint(qids_val), "DATA LEAKAGE: Overlapping questions between FIT and VAL!"
print("Split Integrity Audit: PASSED (Zero paper or question overlap between FIT and VAL).")

TRAIN-FIT Manifest: 1746 questions across 656 papers
INTERNAL-VAL Manifest: 342 questions across 116 papers
Split Integrity Audit: PASSED (Zero paper or question overlap between FIT and VAL).


## Section 4: Target Formulation ($T_3$ Multi-Label Admissibility) Verification

In [4]:
fit_bundle, val_bundle, dev_bundle, _ = build_w5b_splits_and_bundles(save_manifests=False)

# Admissibility rates across budgets k in {2, 4, 5, 6, 8}
fit_t3_rates = fit_bundle["t3_labels"].mean(axis=0)
val_t3_rates = val_bundle["t3_labels"].mean(axis=0)
dev_t3_rates = dev_bundle["t3_labels"].mean(axis=0)

df_target_rates = pd.DataFrame({
    "Budget (k)": K_CLASSES,
    "TRAIN-FIT Positive Rate": np.round(fit_t3_rates, 4),
    "INTERNAL-VAL Positive Rate": np.round(val_t3_rates, 4),
    "DEV Positive Rate": np.round(dev_t3_rates, 4),
})
print("=== T3 MULTI-LABEL ADMISSIBILITY RATES ===")
print(df_target_rates.to_string(index=False))

=== T3 MULTI-LABEL ADMISSIBILITY RATES ===
 Budget (k)  TRAIN-FIT Positive Rate  INTERNAL-VAL Positive Rate  DEV Positive Rate
          2                   0.5974                      0.5585             0.5714
          4                   0.6775                      0.6667             0.6494
          5                   0.7050                      0.7310             0.6537
          6                   0.7245                      0.7193             0.6710
          8                   0.7606                      0.7807             0.7100


## Section 5: MiniLM Encoder Architecture Audit

In [5]:
base_encoder = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
audit_info = audit_minilm_architecture(base_encoder)

print("=== MINILM-L6-V2 ARCHITECTURE AUDIT ===")
print(f"Total Transformer Layers: {audit_info['num_transformer_layers']}")
print(f"Total Base Encoder Parameters: {audit_info['total_encoder_params']:,}")
print(f"Embeddings Parameters: {audit_info['embeddings_param_count']:,}")
print(f"Layer 4 Parameters: {audit_info['layer_4_param_count']:,}")
print(f"Layer 5 Parameters: {audit_info['layer_5_param_count']:,}")
print(f"Pooler exists: {audit_info['has_pooler']}")

=== MINILM-L6-V2 ARCHITECTURE AUDIT ===
Total Transformer Layers: 6
Total Base Encoder Parameters: 22,713,216
Embeddings Parameters: 11,918,592
Layer 4 Parameters: 1,774,464
Layer 5 Parameters: 1,774,464
Pooler exists: True


## Section 6: Capacity Ladder Model Definitions and Parameter Breakdown

In [6]:
model_c0 = AdaptedContentAwareAllocator(capacity_condition="C0")
model_c1 = AdaptedContentAwareAllocator(capacity_condition="C1")
model_c2 = AdaptedContentAwareAllocator(capacity_condition="C2")

breakdowns = [
    model_c0.get_parameter_breakdown(),
    model_c1.get_parameter_breakdown(),
    model_c2.get_parameter_breakdown(),
]

df_params = pd.DataFrame(breakdowns)
print("=== CAPACITY LADDER PARAMETER COUNTS ===")
print(df_params[[
    "capacity_condition",
    "total_params",
    "trainable_params",
    "trainable_pct",
    "encoder_trainable_params",
    "encoder_frozen_params",
    "head_and_aggregator_trainable_params",
]].to_string(index=False))

=== CAPACITY LADDER PARAMETER COUNTS ===
capacity_condition  total_params  trainable_params  trainable_pct  encoder_trainable_params  encoder_frozen_params  head_and_aggregator_trainable_params
                C0      25529861           2816645          11.03                         0               22713216                               2816645
                C1      25529861           4591109          17.98                   1774464               20938752                               2816645
                C2      25529861           6365573          24.93                   3548928               19164288                               2816645


## Section 7: Pairwise Query-Passage Tokenization Audit

In [7]:
from src.week5.data import compute_truncation_statistics

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
trunc_stats = compute_truncation_statistics(dev_bundle, tokenizer, max_seq_length=256)

print("=== TOKENIZATION TRUNCATION AUDIT (DEV) ===")
for k, v in trunc_stats.items():
    print(f"{k}: {v}")

=== TOKENIZATION TRUNCATION AUDIT (DEV) ===
total_query_passage_pairs: 2288
truncated_pairs_count: 608
pct_truncated: 26.57
mean_original_tokens: 207.3
mean_retained_tokens: 203.5
max_seq_length: 256


## Section 8: Execution Mode Configuration

In [8]:
# Master execution flag: Default False for full canonical experiment execution
SMOKE_MODE = False
print(f"SMOKE_MODE = {SMOKE_MODE}")
if SMOKE_MODE:
    print("WARNING: Running in FAST SMOKE MODE (1 epoch, subset of data, seed 42 only).")
else:
    print("Running in FULL CANONICAL EXPERIMENT MODE (All 3 seeds, full data, early stopping).")

SMOKE_MODE = False
Running in FULL CANONICAL EXPERIMENT MODE (All 3 seeds, full data, early stopping).


## Section 9: C1 Training (Last-1 Transformer Block Trainable) Across 3 Seeds

In [9]:
from torch.utils.data import DataLoader, Subset

# Construct full Datasets
train_dataset = ContentAllocatorDataset(fit_bundle, tokenizer=tokenizer, condition="I3", max_seq_length=256)
val_dataset = ContentAllocatorDataset(val_bundle, tokenizer=tokenizer, condition="I3", max_seq_length=256)
dev_dataset = ContentAllocatorDataset(dev_bundle, tokenizer=tokenizer, condition="I3", max_seq_length=256)

c1_seeds_to_run = [PRIMARY_SEED] if SMOKE_MODE else SEEDS
c1_epochs = 1 if SMOKE_MODE else MAX_EPOCHS

models_c1 = {}
histories_c1 = {}
healths_c1 = {}

for seed in c1_seeds_to_run:
    print(f"\n--- Training C1 (Last-1 Layer) | Seed {seed} ---")
    set_seed(seed)
    
    train_loader = DataLoader(
        Subset(train_dataset, range(16)) if SMOKE_MODE else train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        Subset(val_dataset, range(8)) if SMOKE_MODE else val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    
    model = AdaptedContentAwareAllocator(
        capacity_condition="C1",
        base_encoder_name="sentence-transformers/all-MiniLM-L6-v2",
    ).to(DEVICE)
    
    train_res = train_adapted_allocator(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=c1_epochs,
        encoder_lr=ENCODER_LR,
        head_lr=HEAD_LR,
        patience=PATIENCE,
        grad_clip=GRAD_CLIP,
        device=DEVICE,
        verbose=True,
    )
    
    models_c1[seed] = train_res["model"]
    histories_c1[seed] = train_res["history"]
    healths_c1[seed] = train_res["adaptation_health"]
    
    # Save checkpoint
    out_pt = Path(f"results/week5/content_allocator/w5c_encoder_adaptation/models/C1_seed_{seed}.pt")
    out_pt.parent.mkdir(parents=True, exist_ok=True)
    torch.save(train_res["model"].state_dict(), out_pt)
    print(f"Saved C1 Seed {seed} model to {out_pt}")


--- Training C1 (Last-1 Layer) | Seed 42 ---


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
2026-10-11 15:30:09,830 [INFO] Epoch 01/15 | Train Loss: 0.6208 | Val Loss: 0.6080 | Val AUPRC: 0.7218 | Val AUROC: 0.5386 [BEST]
2026-10-11 15:30:37,101 [INFO] Epoch 02/15 | Train Loss: 0.6111 | Val Loss: 0.6064 | Val AUPRC: 0.7293 | Val AUROC: 0.5639 [BEST]
2026-10-11 15:31:04,484 [INFO] Epoch 03/15 | Train Loss: 0.5962 | Val Loss: 0.6069 | Val AUPRC: 0.7349 | Val AUROC: 0.5620 [BEST]
2026-10-11 15:31:32,099 [INFO] Epoch 04/15 | Train Loss: 0.5554 | Val Lo

Saved C1 Seed 42 model to results/week6/encoder_adaptation/models/C1_seed_42.pt

--- Training C1 (Last-1 Layer) | Seed 123 ---


2026-10-11 15:32:57,685 [INFO] Epoch 01/15 | Train Loss: 0.6156 | Val Loss: 0.6082 | Val AUPRC: 0.7236 | Val AUROC: 0.5529 [BEST]
2026-10-11 15:33:27,102 [INFO] Epoch 02/15 | Train Loss: 0.6011 | Val Loss: 0.6060 | Val AUPRC: 0.7221 | Val AUROC: 0.5455
2026-10-11 15:33:56,512 [INFO] Epoch 03/15 | Train Loss: 0.5787 | Val Loss: 0.6054 | Val AUPRC: 0.7214 | Val AUROC: 0.5555
2026-10-11 15:34:26,035 [INFO] Epoch 04/15 | Train Loss: 0.5422 | Val Loss: 0.6455 | Val AUPRC: 0.7188 | Val AUROC: 0.5467
2026-10-11 15:34:26,036 [INFO] Early stopping triggered at epoch 4. Best epoch was 1 (Val AUPRC: 0.7236).


Saved C1 Seed 123 model to results/week6/encoder_adaptation/models/C1_seed_123.pt

--- Training C1 (Last-1 Layer) | Seed 2026 ---


2026-10-11 15:34:55,752 [INFO] Epoch 01/15 | Train Loss: 0.6165 | Val Loss: 0.6088 | Val AUPRC: 0.7124 | Val AUROC: 0.5380 [BEST]
2026-10-11 15:35:25,030 [INFO] Epoch 02/15 | Train Loss: 0.6071 | Val Loss: 0.6037 | Val AUPRC: 0.7263 | Val AUROC: 0.5483 [BEST]
2026-10-11 15:35:53,993 [INFO] Epoch 03/15 | Train Loss: 0.5900 | Val Loss: 0.6280 | Val AUPRC: 0.7233 | Val AUROC: 0.5557
2026-10-11 15:36:22,960 [INFO] Epoch 04/15 | Train Loss: 0.5585 | Val Loss: 0.6271 | Val AUPRC: 0.7378 | Val AUROC: 0.5625 [BEST]
2026-10-11 15:36:51,853 [INFO] Epoch 05/15 | Train Loss: 0.5057 | Val Loss: 0.6648 | Val AUPRC: 0.7212 | Val AUROC: 0.5392
2026-10-11 15:37:20,809 [INFO] Epoch 06/15 | Train Loss: 0.4692 | Val Loss: 0.6781 | Val AUPRC: 0.7230 | Val AUROC: 0.5443
2026-10-11 15:37:49,598 [INFO] Epoch 07/15 | Train Loss: 0.4205 | Val Loss: 0.7164 | Val AUPRC: 0.7163 | Val AUROC: 0.5322
2026-10-11 15:37:49,599 [INFO] Early stopping triggered at epoch 7. Best epoch was 4 (Val AUPRC: 0.7378).


Saved C1 Seed 2026 model to results/week6/encoder_adaptation/models/C1_seed_2026.pt


## Section 10: C2 Training (Last-2 Transformer Blocks Trainable) Across 3 Seeds

In [10]:
c2_seeds_to_run = [PRIMARY_SEED] if SMOKE_MODE else SEEDS
c2_epochs = 1 if SMOKE_MODE else MAX_EPOCHS

models_c2 = {}
histories_c2 = {}
healths_c2 = {}

for seed in c2_seeds_to_run:
    print(f"\n--- Training C2 (Last-2 Layers) | Seed {seed} ---")
    set_seed(seed)
    
    train_loader = DataLoader(
        Subset(train_dataset, range(16)) if SMOKE_MODE else train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        Subset(val_dataset, range(8)) if SMOKE_MODE else val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    
    model = AdaptedContentAwareAllocator(
        capacity_condition="C2",
        base_encoder_name="sentence-transformers/all-MiniLM-L6-v2",
    ).to(DEVICE)
    
    train_res = train_adapted_allocator(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=c2_epochs,
        encoder_lr=ENCODER_LR,
        head_lr=HEAD_LR,
        patience=PATIENCE,
        grad_clip=GRAD_CLIP,
        device=DEVICE,
        verbose=True,
    )
    
    models_c2[seed] = train_res["model"]
    histories_c2[seed] = train_res["history"]
    healths_c2[seed] = train_res["adaptation_health"]
    
    # Save checkpoint
    out_pt = Path(f"results/week5/content_allocator/w5c_encoder_adaptation/models/C2_seed_{seed}.pt")
    out_pt.parent.mkdir(parents=True, exist_ok=True)
    torch.save(train_res["model"].state_dict(), out_pt)
    print(f"Saved C2 Seed {seed} model to {out_pt}")


--- Training C2 (Last-2 Layers) | Seed 42 ---


2026-10-11 15:38:24,404 [INFO] Epoch 01/15 | Train Loss: 0.6209 | Val Loss: 0.6081 | Val AUPRC: 0.7230 | Val AUROC: 0.5410 [BEST]
2026-10-11 15:38:58,449 [INFO] Epoch 02/15 | Train Loss: 0.6108 | Val Loss: 0.5969 | Val AUPRC: 0.7716 | Val AUROC: 0.6122 [BEST]
2026-10-11 15:39:32,838 [INFO] Epoch 03/15 | Train Loss: 0.5824 | Val Loss: 0.5809 | Val AUPRC: 0.8028 | Val AUROC: 0.6514 [BEST]
2026-10-11 15:40:07,128 [INFO] Epoch 04/15 | Train Loss: 0.5395 | Val Loss: 0.5797 | Val AUPRC: 0.7998 | Val AUROC: 0.6495
2026-10-11 15:40:41,129 [INFO] Epoch 05/15 | Train Loss: 0.5114 | Val Loss: 0.6013 | Val AUPRC: 0.7882 | Val AUROC: 0.6331
2026-10-11 15:41:15,349 [INFO] Epoch 06/15 | Train Loss: 0.4645 | Val Loss: 0.6572 | Val AUPRC: 0.7732 | Val AUROC: 0.6134
2026-10-11 15:41:15,349 [INFO] Early stopping triggered at epoch 6. Best epoch was 3 (Val AUPRC: 0.8028).


Saved C2 Seed 42 model to results/week6/encoder_adaptation/models/C2_seed_42.pt

--- Training C2 (Last-2 Layers) | Seed 123 ---


2026-10-11 15:41:49,915 [INFO] Epoch 01/15 | Train Loss: 0.6157 | Val Loss: 0.6082 | Val AUPRC: 0.7278 | Val AUROC: 0.5598 [BEST]
2026-10-11 15:42:23,985 [INFO] Epoch 02/15 | Train Loss: 0.5981 | Val Loss: 0.5875 | Val AUPRC: 0.7676 | Val AUROC: 0.6129 [BEST]
2026-10-11 15:42:58,057 [INFO] Epoch 03/15 | Train Loss: 0.5649 | Val Loss: 0.5842 | Val AUPRC: 0.7940 | Val AUROC: 0.6393 [BEST]
2026-10-11 15:43:32,140 [INFO] Epoch 04/15 | Train Loss: 0.5314 | Val Loss: 0.6055 | Val AUPRC: 0.7736 | Val AUROC: 0.6169
2026-10-11 15:44:06,324 [INFO] Epoch 05/15 | Train Loss: 0.4861 | Val Loss: 0.6267 | Val AUPRC: 0.7695 | Val AUROC: 0.6114
2026-10-11 15:44:40,444 [INFO] Epoch 06/15 | Train Loss: 0.4414 | Val Loss: 0.6427 | Val AUPRC: 0.7579 | Val AUROC: 0.5971
2026-10-11 15:44:40,445 [INFO] Early stopping triggered at epoch 6. Best epoch was 3 (Val AUPRC: 0.7940).


Saved C2 Seed 123 model to results/week6/encoder_adaptation/models/C2_seed_123.pt

--- Training C2 (Last-2 Layers) | Seed 2026 ---


2026-10-11 15:45:15,261 [INFO] Epoch 01/15 | Train Loss: 0.6166 | Val Loss: 0.6087 | Val AUPRC: 0.7156 | Val AUROC: 0.5410 [BEST]
2026-10-11 15:45:49,324 [INFO] Epoch 02/15 | Train Loss: 0.6065 | Val Loss: 0.5963 | Val AUPRC: 0.7718 | Val AUROC: 0.5985 [BEST]
2026-10-11 15:46:23,637 [INFO] Epoch 03/15 | Train Loss: 0.5772 | Val Loss: 0.6019 | Val AUPRC: 0.7973 | Val AUROC: 0.6383 [BEST]
2026-10-11 15:46:57,792 [INFO] Epoch 04/15 | Train Loss: 0.5406 | Val Loss: 0.5958 | Val AUPRC: 0.7957 | Val AUROC: 0.6405
2026-10-11 15:47:31,984 [INFO] Epoch 05/15 | Train Loss: 0.4998 | Val Loss: 0.6154 | Val AUPRC: 0.7728 | Val AUROC: 0.6036
2026-10-11 15:48:06,233 [INFO] Epoch 06/15 | Train Loss: 0.4571 | Val Loss: 0.6451 | Val AUPRC: 0.7864 | Val AUROC: 0.6149
2026-10-11 15:48:06,234 [INFO] Early stopping triggered at epoch 6. Best epoch was 3 (Val AUPRC: 0.7973).


Saved C2 Seed 2026 model to results/week6/encoder_adaptation/models/C2_seed_2026.pt


## Section 11: INTERNAL-VAL Performance and Threshold Calibration

In [11]:
val_loader_full = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

tau_records = []
selected_taus = {}

# Evaluate on INTERNAL-VAL and select tau for each model
for cond, models_dict in [("C1", models_c1), ("C2", models_c2)]:
    for seed, m in models_dict.items():
        m.eval()
        all_probs = []
        with torch.no_grad():
            for batch in val_loader_full:
                logits = m(batch)["admissibility_logits"]
                all_probs.append(torch.sigmoid(logits).cpu().numpy())
        val_probs = np.concatenate(all_probs, axis=0)
        
        tau_res = select_tau_w5c(
            val_supervision_df=val_bundle["supervision_df"],
            val_pred_probs=val_probs,
            val_tokens_map=val_bundle["tokens_map"],
            tau_grid=TAU_GRID,
            quality_tolerance=TAU_TOLERANCE,
        )
        
        selected_taus[(cond, seed)] = tau_res["selected_tau"]
        tau_records.append({
            "capacity_condition": cond,
            "seed": seed,
            "selected_tau": tau_res["selected_tau"],
            "tau_boundary_hit": tau_res["tau_boundary_hit"],
            "quality_constraint_met": tau_res["constraint_met"],
            "static_k8_val_f1": round(tau_res["static_k8_f1"], 4),
            "selected_val_f1": round(tau_res["selected_result"]["mean_answer_f1"], 4),
            "selected_val_tokens": round(tau_res["selected_result"]["mean_context_tokens"], 1),
        })

df_tau_sel = pd.DataFrame(tau_records)
df_tau_sel.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/tables/w5c_tau_selection.csv", index=False)
print("=== THRESHOLD CALIBRATION ON INTERNAL-VAL ===")
print(df_tau_sel.to_string(index=False))

=== THRESHOLD CALIBRATION ON INTERNAL-VAL ===
capacity_condition  seed  selected_tau  tau_boundary_hit  quality_constraint_met  static_k8_val_f1  selected_val_f1  selected_val_tokens
                C1    42           0.7              True                   False            0.4457           0.4114               1315.5
                C1   123           0.7              True                   False            0.4457           0.3840               1170.0
                C1  2026           0.7              True                   False            0.4457           0.3809                991.8
                C2    42           0.7              True                   False            0.4457           0.4179               1307.3
                C2   123           0.7              True                   False            0.4457           0.3933               1146.6
                C2  2026           0.7              True                   False            0.4457           0.4283               14

## Section 12: Adaptation Health Check & Representation Shift Analysis

In [12]:
# 1. Save Gradient & Adaptation Health Check
all_health = {
    "C1": {str(seed): healths_c1[seed] for seed in healths_c1},
    "C2": {str(seed): healths_c2[seed] for seed in healths_c2},
}
with open("results/week5/content_allocator/w5c_encoder_adaptation/diagnostics/representation_adaptation_health.json", "w", encoding="utf-8") as f:
    json.dump(all_health, f, indent=2)

# 2. Compute Representation Shift on INTERNAL-VAL
shift_records = []
for cond, models_dict in [("C1", models_c1), ("C2", models_c2)]:
    for seed, m in models_dict.items():
        shift_dict = compute_representation_shift(
            pretrained_model=model_c0,
            adapted_model=m,
            val_loader=val_loader_full,
            max_batches=5,
            device=DEVICE,
        )
        shift_dict["capacity_condition"] = cond
        shift_dict["seed"] = seed
        shift_records.append(shift_dict)

df_rep_shift = pd.DataFrame(shift_records)
df_rep_shift.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/diagnostics/representation_shift.csv", index=False)

print("=== REPRESENTATION SHIFT DIAGNOSTICS ===")
print(df_rep_shift[[
    "capacity_condition",
    "seed",
    "mean_cosine_similarity",
    "mean_representation_shift_norm",
    "within_query_passage_var_adapted",
    "across_query_var_adapted",
]].to_string(index=False))

=== REPRESENTATION SHIFT DIAGNOSTICS ===
capacity_condition  seed  mean_cosine_similarity  mean_representation_shift_norm  within_query_passage_var_adapted  across_query_var_adapted
                C1    42                  0.9823                          0.4009                            1.5451                    1.7151
                C1   123                  0.9922                          0.2625                            1.5431                    1.7191
                C1  2026                  0.9767                          0.4608                            1.5554                    1.7290
                C2    42                  0.9171                          0.8446                            1.2785                    1.6612
                C2   123                  0.9286                          0.7849                            1.3177                    1.6935
                C2  2026                  0.9202                          0.8291                            1.270

## Section 13: Final Evaluation on Frozen DEV Set

In [13]:
dev_loader_full = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
dev_sup_df = dev_bundle["supervision_df"]

eval_results_dev = {}
predictions_dict = {
    "question_id": dev_bundle["question_ids"],
    "paper_id": dev_bundle["paper_ids"],
}

# Add W5-B C0 reference predictions if available
if c0_ref["predictions_df"] is not None:
    for seed in SEEDS:
        col = f"pred_k_I3_seed{seed}"
        if col in c0_ref["predictions_df"]:
            predictions_dict[f"pred_k_C0_seed{seed}"] = c0_ref["predictions_df"][col]

for cond, models_dict in [("C1", models_c1), ("C2", models_c2)]:
    for seed, m in models_dict.items():
        tau = selected_taus[(cond, seed)]
        eval_res = evaluate_w5c_model_on_dev(
            model=m,
            dev_loader=dev_loader_full,
            dev_sup_df=dev_sup_df,
            tau=tau,
            device=DEVICE,
        )
        eval_results_dev[(cond, seed)] = eval_res
        predictions_dict[f"pred_k_{cond}_seed{seed}"] = eval_res["preds_k"]

df_dev_preds = pd.DataFrame(predictions_dict)
df_dev_preds.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/predictions/dev_predictions.csv", index=False)
print("Saved DEV predictions to results/week5/content_allocator/w5c_encoder_adaptation/predictions/dev_predictions.csv")

Saved DEV predictions to results/week6/encoder_adaptation/predictions/dev_predictions.csv


## Section 14: Main Policy Table across Capacity Ladder and Baselines

In [14]:
# Compile Main Results Table
main_rows = []

# Baselines
w4_eval = load_w4_baseline_dev(dev_sup_df=dev_sup_df)

main_rows.append({
    "model_name": "Quality Oracle",
    "mean_answer_f1": 0.4895,
    "std_answer_f1": 0.0,
    "mean_selected_k": 3.53,
    "mean_context_tokens": 1021.5,
    "token_reduction_pct": 41.65,
    "mean_oracle_regret": 0.0000,
    "fallback_rate_pct": 0.0,
    "recovered_oracle_headroom": 1.0,
})

main_rows.append({
    "model_name": "Cost-Aware Epsilon Oracle",
    "mean_answer_f1": 0.4894,
    "std_answer_f1": 0.0,
    "mean_selected_k": 3.49,
    "mean_context_tokens": 839.4,
    "token_reduction_pct": 52.05,
    "mean_oracle_regret": 0.0001,
    "fallback_rate_pct": 0.0,
    "recovered_oracle_headroom": 0.9989,
})

main_rows.append({
    "model_name": "Best Static SARA (k=8)",
    "mean_answer_f1": 0.3950,
    "std_answer_f1": 0.0,
    "mean_selected_k": 8.00,
    "mean_context_tokens": 1750.5,
    "token_reduction_pct": 0.0,
    "mean_oracle_regret": 0.0945,
    "fallback_rate_pct": 0.0,
    "recovered_oracle_headroom": 0.0,
})

main_rows.append({
    "model_name": "Static k=6",
    "mean_answer_f1": 0.3857,
    "std_answer_f1": 0.0,
    "mean_selected_k": 6.00,
    "mean_context_tokens": 1351.1,
    "token_reduction_pct": 22.82,
    "mean_oracle_regret": 0.1038,
    "fallback_rate_pct": 0.0,
    "recovered_oracle_headroom": -0.0984,
})

main_rows.append({
    "model_name": "W4 Balanced Logistic Regression",
    "mean_answer_f1": w4_eval["mean_answer_f1"],
    "std_answer_f1": 0.0,
    "mean_selected_k": w4_eval["mean_selected_k"],
    "mean_context_tokens": w4_eval["mean_context_tokens"],
    "token_reduction_pct": w4_eval["token_reduction_pct_vs_k8"],
    "mean_oracle_regret": w4_eval["mean_oracle_regret"],
    "fallback_rate_pct": 0.0,
    "recovered_oracle_headroom": w4_eval["recovered_oracle_headroom"],
})

# C0 Frozen MiniLM (W5-B Reference)
main_rows.append({
    "model_name": "C0 Frozen MiniLM",
    "mean_answer_f1": c0_ref["mean_answer_f1"],
    "std_answer_f1": c0_ref.get("std_answer_f1", 0.0090),
    "mean_selected_k": c0_ref["mean_selected_k"],
    "mean_context_tokens": c0_ref["mean_context_tokens"],
    "token_reduction_pct": c0_ref["token_reduction_pct"],
    "mean_oracle_regret": 0.1085,
    "fallback_rate_pct": c0_ref["fallback_rate_pct"],
    "recovered_oracle_headroom": c0_ref["recovered_oracle_headroom"],
})

# C1 and C2 aggregates
for cond in ["C1", "C2"]:
    cond_evals = [eval_results_dev[(cond, seed)]["policy_metrics"] for seed in SEEDS if (cond, seed) in eval_results_dev]
    f1s = [e["mean_answer_f1"] for e in cond_evals]
    ks = [e["mean_selected_k"] for e in cond_evals]
    toks = [e["mean_context_tokens"] for e in cond_evals]
    reducs = [e["token_reduction_pct_vs_k8"] for e in cond_evals]
    regs = [e["mean_oracle_regret"] for e in cond_evals]
    rohs = [e["recovered_oracle_headroom"] for e in cond_evals]
    fbs = [eval_results_dev[(cond, seed)]["fallback_rate_pct"] for seed in SEEDS if (cond, seed) in eval_results_dev]
    
    label = "C1 Last-1 Layer Trainable" if cond == "C1" else "C2 Last-2 Layers Trainable"
    main_rows.append({
        "model_name": label,
        "mean_answer_f1": round(float(np.mean(f1s)), 4),
        "std_answer_f1": round(float(np.std(f1s)), 4) if len(f1s) > 1 else 0.0,
        "mean_selected_k": round(float(np.mean(ks)), 2),
        "mean_context_tokens": round(float(np.mean(toks)), 1),
        "token_reduction_pct": round(float(np.mean(reducs)), 2),
        "mean_oracle_regret": round(float(np.mean(regs)), 4),
        "fallback_rate_pct": round(float(np.mean(fbs)), 2),
        "recovered_oracle_headroom": round(float(np.mean(rohs)), 4),
    })

df_main = pd.DataFrame(main_rows)
df_main.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/tables/w5c_main_results.csv", index=False)
print("=== MAIN POLICY RESULTS TABLE ===")
print(df_main.to_string(index=False))

=== MAIN POLICY RESULTS TABLE ===
                     model_name  mean_answer_f1  std_answer_f1  mean_selected_k  mean_context_tokens  token_reduction_pct  mean_oracle_regret  fallback_rate_pct  recovered_oracle_headroom
                 Quality Oracle        0.489500       0.000000         3.530000          1021.500000            41.650000              0.0000           0.000000                     1.0000
      Cost-Aware Epsilon Oracle        0.489400       0.000000         3.490000           839.400000            52.050000              0.0001           0.000000                     0.9989
         Best Static SARA (k=8)        0.395000       0.000000         8.000000          1750.500000             0.000000              0.0945           0.000000                     0.0000
                     Static k=6        0.385700       0.000000         6.000000          1351.100000            22.820000              0.1038           0.000000                    -0.0984
W4 Balanced Logistic Regre

## Section 15: Seed Stability and Per-Seed Breakdown

In [15]:
seed_rows = []
for (cond, seed), res in eval_results_dev.items():
    p = res["policy_metrics"]
    m = res["multilabel_metrics"]
    dist = res["k_distribution"]
    seed_rows.append({
        "capacity_condition": cond,
        "seed": seed,
        "selected_tau": res["selected_tau"],
        "dev_answer_f1": p["mean_answer_f1"],
        "dev_mean_k": p["mean_selected_k"],
        "dev_context_tokens": p["mean_context_tokens"],
        "token_reduction_pct": p["token_reduction_pct_vs_k8"],
        "mean_oracle_regret": p["mean_oracle_regret"],
        "zero_regret_pct": p["zero_regret_pct"],
        "fallback_rate_pct": res["fallback_rate_pct"],
        "recovered_oracle_headroom": p["recovered_oracle_headroom"],
        "macro_auprc": m["macro_auprc"],
        "macro_auroc": m["macro_auroc"],
        "logit_variance": res["logit_variance"],
        **dist,
    })

df_seeds = pd.DataFrame(seed_rows)
df_seeds.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/tables/w5c_seed_results.csv", index=False)
print("=== PER-SEED POLICY RESULTS ===")
print(df_seeds[[
    "capacity_condition",
    "seed",
    "dev_answer_f1",
    "dev_mean_k",
    "dev_context_tokens",
    "token_reduction_pct",
    "fallback_rate_pct",
    "macro_auprc",
]].to_string(index=False))

=== PER-SEED POLICY RESULTS ===
capacity_condition  seed  dev_answer_f1  dev_mean_k  dev_context_tokens  token_reduction_pct  fallback_rate_pct  macro_auprc
                C1    42         0.3741        5.91              1326.9                24.20              42.86       0.7211
                C1   123         0.3660        4.91              1122.8                35.86               0.00       0.6785
                C1  2026         0.3459        4.44              1026.9                41.34              14.72       0.7308
                C2    42         0.3758        6.05              1354.0                22.65              59.74       0.7856
                C2   123         0.3580        5.31              1203.8                31.23              28.14       0.7615
                C2  2026         0.3891        6.55              1454.9                16.89              47.19       0.7776


## Section 16: Selected Budget Distribution ($k \in \{2, 4, 5, 6, 8\}$)

In [16]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

plot_data = []
for cond in ["C1", "C2"]:
    c_df = df_seeds[df_seeds["capacity_condition"] == cond]
    if len(c_df) > 0:
        row = {f"k={k}": c_df[f"pct_k{k}"].mean() for k in K_CLASSES}
        row["Condition"] = cond
        plot_data.append(row)

if plot_data:
    df_plot_k = pd.DataFrame(plot_data).set_index("Condition")
    df_plot_k.plot(kind="bar", stacked=True, ax=ax, colormap="viridis", width=0.5)
    plt.ylabel("Percentage of DEV Queries (%)", fontweight="bold")
    plt.title("Budget Allocation Distribution across Capacity Conditions", fontweight="bold")
    plt.legend(title="Budget", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("results/week5/content_allocator/w5c_encoder_adaptation/figures/w5c_selected_k_distribution.png")
    plt.close()
    print("Saved selected-k distribution plot to results/week5/content_allocator/w5c_encoder_adaptation/figures/w5c_selected_k_distribution.png")

Saved selected-k distribution plot to results/week6/encoder_adaptation/figures/w6_selected_k_distribution.png


## Section 17: Secondary Multi-Label Diagnostic Metrics

In [17]:
multi_rows = []
for (cond, seed), res in eval_results_dev.items():
    m = res["multilabel_metrics"]
    multi_rows.append({
        "capacity_condition": cond,
        "seed": seed,
        "macro_auprc": m["macro_auprc"],
        "macro_auroc": m["macro_auroc"],
        "micro_auprc": m["micro_auprc"],
        "overall_brier": m["overall_brier"],
        "auprc_k2": m["per_budget_auprc"]["k2"],
        "auprc_k4": m["per_budget_auprc"]["k4"],
        "auprc_k5": m["per_budget_auprc"]["k5"],
        "auprc_k6": m["per_budget_auprc"]["k6"],
        "auprc_k8": m["per_budget_auprc"]["k8"],
    })

df_multi = pd.DataFrame(multi_rows)
df_multi.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/tables/w5c_multilabel_metrics.csv", index=False)
print("=== MULTI-LABEL METRICS TABLE ===")
print(df_multi[[
    "capacity_condition",
    "seed",
    "macro_auprc",
    "macro_auroc",
    "micro_auprc",
    "overall_brier",
]].to_string(index=False))

=== MULTI-LABEL METRICS TABLE ===
capacity_condition  seed  macro_auprc  macro_auroc  micro_auprc  overall_brier
                C1    42       0.7211       0.5819       0.7188         0.2213
                C1   123       0.6785       0.5404       0.6892         0.2254
                C1  2026       0.7308       0.5956       0.7297         0.2286
                C2    42       0.7856       0.6435       0.7876         0.2119
                C2   123       0.7615       0.6235       0.7616         0.2209
                C2  2026       0.7776       0.6364       0.7811         0.2192


## Section 18: Quality–Cost Pareto Frontier Visualization

In [18]:
plot_w5c_pareto_curve(
    results_df=df_main,
    output_path="results/week5/content_allocator/w5c_encoder_adaptation/figures/w5c_pareto.png",
)
print("Pareto frontier figure saved to results/week5/content_allocator/w5c_encoder_adaptation/figures/w5c_pareto.png")

2026-10-11 15:48:53,832 [INFO] Saved Pareto plot to results/week6/encoder_adaptation/figures/w6_pareto.png


Pareto frontier figure saved to results/week6/encoder_adaptation/figures/w6_pareto.png


## Section 19: Paper-Clustered Statistical Bootstrap Analysis

In [19]:
from src.week5.evaluation import evaluate_allocation_policy

paper_ids = dev_bundle["paper_ids"]

# Compute across-seed mean per-query vectors for C1 and C2
c1_query_f1s = np.mean([eval_results_dev[("C1", s)]["policy_metrics"]["per_query_f1s"] for s in c1_seeds_to_run], axis=0)
c1_query_toks = np.mean([eval_results_dev[("C1", s)]["policy_metrics"]["per_query_tokens"] for s in c1_seeds_to_run], axis=0)

c2_query_f1s = np.mean([eval_results_dev[("C2", s)]["policy_metrics"]["per_query_f1s"] for s in c2_seeds_to_run], axis=0)
c2_query_toks = np.mean([eval_results_dev[("C2", s)]["policy_metrics"]["per_query_tokens"] for s in c2_seeds_to_run], axis=0)

# Load C0 per-query vectors from W5-B predictions
if c0_ref["predictions_df"] is not None:
    c0_query_f1s_list = []
    c0_query_toks_list = []
    for s in SEEDS:
        col = f"pred_k_I3_seed{s}"
        if col in c0_ref["predictions_df"]:
            e = evaluate_allocation_policy(dev_sup_df, c0_ref["predictions_df"][col].values)
            c0_query_f1s_list.append(e["per_query_f1s"])
            c0_query_toks_list.append(e["per_query_tokens"])
    c0_query_f1s = np.mean(c0_query_f1s_list, axis=0)
    c0_query_toks = np.mean(c0_query_toks_list, axis=0)
else:
    c0_query_f1s = np.full(len(dev_sup_df), C0_REF_F1)
    c0_query_toks = np.full(len(dev_sup_df), C0_REF_TOKENS)

eval_c0_dict = {"per_query_f1s": c0_query_f1s, "per_query_tokens": c0_query_toks}
eval_c1_dict = {"per_query_f1s": c1_query_f1s, "per_query_tokens": c1_query_toks}
eval_c2_dict = {"per_query_f1s": c2_query_f1s, "per_query_tokens": c2_query_toks}
eval_k8_dict = {"per_query_f1s": dev_sup_df["f1_k8"].values, "per_query_tokens": np.full(len(dev_sup_df), 1750.5)}

n_boot = 100 if SMOKE_MODE else 2000
df_boot = run_paper_clustered_bootstrap_comparisons(
    paper_ids=paper_ids,
    eval_c0=eval_c0_dict,
    eval_c1=eval_c1_dict,
    eval_c2=eval_c2_dict,
    eval_w4=w4_eval,
    eval_k8=eval_k8_dict,
    n_bootstrap=n_boot,
    seed=42,
)

df_boot.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/bootstrap/w5c_bootstrap_comparisons.csv", index=False)
print("=== PAPER-CLUSTERED BOOTSTRAP RESULTS (ACROSS-SEED MEAN VECTORS) ===")
print(df_boot[[
    "comparison",
    "observed_delta_f1",
    "ci_lower_95_f1",
    "ci_upper_95_f1",
    "p_value_f1",
    "significant_f1",
    "observed_delta_tokens",
    "ci_lower_95_tokens",
    "ci_upper_95_tokens",
]].to_string(index=False))

=== PAPER-CLUSTERED BOOTSTRAP RESULTS (ACROSS-SEED MEAN VECTORS) ===
                                           comparison  observed_delta_f1  ci_lower_95_f1  ci_upper_95_f1  p_value_f1  significant_f1  observed_delta_tokens  ci_lower_95_tokens  ci_upper_95_tokens
                         C1 Last-1 Layer vs C0 Frozen            -0.0191         -0.0377         -0.0031      0.0085            True                  -98.6              -147.9               -47.9
                        C2 Last-2 Layers vs C0 Frozen            -0.0068         -0.0312          0.0175      0.2895           False                   80.1                20.2               137.7
                  C2 Last-2 Layers vs C1 Last-1 Layer             0.0123         -0.0055          0.0306      0.0905           False                  178.7               130.6               228.8
Best Adapted (C2 Last-2 Layers) vs W4 Balanced LogReg            -0.0099         -0.0454          0.0263      0.2870           False               

## Section 20: Scientific Capacity Interpretation

Here we synthesize empirical evidence across the capacity ladder to address the five pre-registered questions:
1. **Does task-specific adaptation of the semantic encoder improve allocation quality over frozen MiniLM?**
2. **Is adapting a single Transformer block ($C_1$) sufficient to achieve significant gains?**
3. **Does adapting two Transformer blocks ($C_2$) provide additional benefit over $C_1$?**
4. **Does adapted semantic allocation outperform the handcrafted Week 4 baseline ($W_4$)?**
5. **Does semantic capacity remain the primary bottleneck on the quality–context frontier?**

In [20]:
# Compile capacity comparison table
cap_rows = [
    {
        "Condition": "C0 Frozen MiniLM",
        "Trainable Encoder Layers": 0,
        "Trainable Params": "2.82M (11.0%)",
        "DEV Answer F1": f"{c0_ref['mean_answer_f1']:.4f}",
        "Context Tokens": f"{c0_ref['mean_context_tokens']:.1f}",
        "Token Reduction": f"{c0_ref['token_reduction_pct']:.1f}%",
        "ROH": f"{c0_ref['recovered_oracle_headroom']:.4f}",
    },
]

for cond, label in [("C1", "C1 Last-1 Layer"), ("C2", "C2 Last-2 Layers")]:
    c_df = df_main[df_main["model_name"].str.startswith(cond)]
    if len(c_df) > 0:
        row = c_df.iloc[0]
        n_layers = 1 if cond == "C1" else 2
        pct = "18.0%" if cond == "C1" else "24.9%"
        cap_rows.append({
            "Condition": label,
            "Trainable Encoder Layers": n_layers,
            "Trainable Params": f"{row['mean_answer_f1']:.4f}" if False else f"{df_params[df_params['capacity_condition'] == cond]['trainable_params'].values[0] / 1e6:.2f}M ({pct})",
            "DEV Answer F1": f"{row['mean_answer_f1']:.4f}",
            "Context Tokens": f"{row['mean_context_tokens']:.1f}",
            "Token Reduction": f"{row['token_reduction_pct']:.1f}%",
            "ROH": f"{row['recovered_oracle_headroom']:.4f}",
        })

df_cap_comp = pd.DataFrame(cap_rows)
df_cap_comp.to_csv("results/week5/content_allocator/w5c_encoder_adaptation/tables/w5c_capacity_comparison.csv", index=False)
print("=== CAPACITY LADDER COMPARISON SUMMARY ===")
print(df_cap_comp.to_string(index=False))

=== CAPACITY LADDER COMPARISON SUMMARY ===
       Condition  Trainable Encoder Layers Trainable Params DEV Answer F1 Context Tokens Token Reduction     ROH
C0 Frozen MiniLM                         0    2.82M (11.0%)        0.3810         1258.2           28.1% -0.1475
 C1 Last-1 Layer                         1    4.59M (18.0%)        0.3620         1158.9           33.8% -0.3491
C2 Last-2 Layers                         2    6.37M (24.9%)        0.3743         1337.6           23.6% -0.2192


## Section 21: Pre-Registered Decision Rules for Week 6 (Passage Routing)

Based on empirical performance across the capacity ladder:
- **Observed Finding:** Neither $C_1$ ($F_1 = 0.3620$) nor $C_2$ ($F_1 = 0.3743$) outperformed frozen $C_0$ ($F_1 = 0.3810$) or $W_4$ Balanced LogReg ($F_1 = 0.3842$).
- **Capacity Conclusion:** Lightweight encoder adaptation of 1–2 layers in MiniLM does not surpass the frozen representation or the handcrafted baseline.
- **Decision for Week 6:** Freeze the semantic allocator and transition to **Query-Conditioned Passage Fidelity Routing** (full, medium, compressed passage representation).

In [21]:
print("Week 5-C Notebook execution pipeline complete.")
print("Generated Outputs:")
print(" - tables/w5c_main_results.csv")
print(" - tables/w5c_seed_results.csv")
print(" - tables/w5c_multilabel_metrics.csv")
print(" - tables/w5c_tau_selection.csv")
print(" - tables/w5c_capacity_comparison.csv")
print(" - predictions/dev_predictions.csv")
print(" - bootstrap/w5c_bootstrap_comparisons.csv")
print(" - diagnostics/representation_adaptation_health.json")
print(" - diagnostics/representation_shift.csv")
print(" - figures/w5c_pareto.png")
print(" - figures/w5c_selected_k_distribution.png")

Week 6 Notebook execution pipeline complete.
Generated Outputs:
 - tables/w6_main_results.csv
 - tables/w6_seed_results.csv
 - tables/w6_multilabel_metrics.csv
 - tables/w6_tau_selection.csv
 - tables/w6_capacity_comparison.csv
 - predictions/dev_predictions.csv
 - bootstrap/w6_bootstrap_comparisons.csv
 - diagnostics/representation_adaptation_health.json
 - diagnostics/representation_shift.csv
 - figures/w6_pareto.png
 - figures/w6_selected_k_distribution.png
